# 分词器（Tokenizer）模块总览

源码导航：[`core/tokenizer/`](../../../core/tokenizer/)。

分词器将任意文本映射为离散 token 序列，是语言模型输入流水线的第一步。词表大小、分词粒度、OOV 处理策略对模型的跨语言能力、上下文利用率及词表效率均有直接影响。

## 1. 算法对比

| 算法 | 代表模型 | 合并/切分策略 | OOV 处理 | 子笔记本 |
|---|---|---|---|---|
| **BPE（字符级）** | — | $\arg\max$ 频次 pair 迭代合并 | 回退到字符 | [bpe.ipynb](bpe.ipynb) |
| **Byte-level BPE** | GPT-2/3, LLaMA-2 | 256 字节为初始词表，BPE 合并 | 无 OOV | [byte_level_bpe.ipynb](byte_level_bpe.ipynb) |
| **WordPiece** | BERT, DistilBERT | $\arg\max$ PMI Score 迭代合并 | 整词 → `[UNK]` | [wordpiece.ipynb](wordpiece.ipynb) |
| **Unigram LM** | LLaMA-1, Qwen, T5 | EM + 迭代裁剪，Viterbi 解码 | 无 OOV | [unigram.ipynb](unigram.ipynb) |

## 2. 关键性质对比

- **训练方向**：BPE / WordPiece 自底向上合并；Unigram LM 自顶向下裁剪。
- **编码策略**：BPE/WordPiece 贪心（局部最优）；Unigram LM 使用 Viterbi 动态规划求全局最优切分。
- **词表初始化**：BPE 从字符集合出发；BBPE 从 256 字节出发；Unigram 从高频子串超集出发。
- **OOV**：字符级 BPE 在遇到训练时未见的字符时返回 `<unk>`；BBPE 和 Unigram（字节或字符全覆盖）均不存在 OOV。

## 3. 本项目集成

所有分词器均继承 `BaseTokenizer`（[`core/tokenizer/base.py`](../../../core/tokenizer/base.py)），暴露统一接口：`train(corpus)` / `encode(text)` / `decode(ids)` / `save(path)` / `load(path)`。

In [ ]:
import sys
import os
# 将项目根目录加入 path (假设在 docs/modules/tokenizer 下运行)
sys.path.append(os.path.abspath("../../"))

from core.tokenizer import build_tokenizer

# 1. 创建工厂实例
tokenizer = build_tokenizer("byte_bpe", vocab_size=256 + 100)

# 2. 准备语料
corpus = [
    "I love deep learning.",
    "BPE is a simple but powerful algorithm.",
    "Tokenizer is the first step of LLM."
]

# 3. 训练
tokenizer.train(corpus, verbose=True)

# 4. 编码与解码
text = "I love BPE Tokenizer!"
ids = tokenizer.encode(text)
decoded = tokenizer.decode(ids)

print(f"\n原文: {text}")
print(f"IDs: {ids}")
print(f"解码结果: {decoded}")

## 4. 后续步骤建议

1. **零基础入门**：先读 [01 · BPE（教学版）](bpe.ipynb) 弄懂核心循环。
2. **理解主流 GPT**：阅读 [02 · Byte-level BPE](byte_level_bpe.ipynb) 理解为何没有 `<UNK>`。
3. **理解 LLaMA/Qwen**：阅读 [04 · Unigram LM](unigram.ipynb) 学习目前大模型最主流的实现方式。

---
> [index.ipynb](index.ipynb) 是分词器模块的文档入口。